In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# QUICK-PDE: O Funcție Qiskit de la ColibriTD
> **Note:** Funcțiile Qiskit sunt o caracteristică experimentală disponibilă utilizatorilor planurilor IBM Quantum&reg; Premium, Flex și On-Prem (prin IBM Quantum Platform API). Acestea se află în stare de previzualizare și pot suferi modificări.
## Prezentare generală
Rezolvatorul de Ecuații cu Derivate Parțiale (PDE) prezentat aici face parte din platforma noastră Quantum Innovative Computing Kit (QUICK) (QUICK-PDE) și este ambalat ca o Funcție Qiskit. Cu funcția QUICK-PDE, poți rezolva ecuații cu derivate parțiale specifice unui domeniu pe QPU-urile IBM Quantum. Această funcție se bazează pe algoritmul descris în [lucrarea de prezentare H-DES a ColibriTD.](https://arxiv.org/abs/2410.01130) Acest algoritm poate rezolva probleme complexe de multi-fizică, începând cu Dinamica Computațională a Fluidelor (CFD) și Deformarea Materialelor (MD), urmând să fie adăugate și alte cazuri de utilizare în curând.

Pentru a aborda ecuațiile diferențiale, soluțiile de test sunt codificate ca combinații liniare de funcții ortogonale (de obicei polinoame Chebyshev, și mai specific $2^n$ dintre ele, unde $n$ este numărul de qubiți care codifică funcția ta), parametrizate prin unghiurile unui Circuit Cuantic Variabil (VQC). Ansatz-ul generează o stare care codifică funcția, evaluată prin observabile ale căror combinații permit evaluarea funcției în toate punctele. Poți apoi evalua funcția de pierdere în care sunt codificate ecuațiile diferențiale și ajusta fin unghiurile într-un ciclu hibrid, după cum este prezentat în continuare. Soluțiile de test se apropie treptat de soluțiile reale până când obții un rezultat satisfăcător.

![Fluxul de lucru al funcției QUICK-PDE](../docs/images/guides/colibritd-equation-solver/diagram.svg)

Pe lângă acest ciclu hibrid, poți de asemenea înlănțui diferiți optimizatori. Acest lucru este util atunci când dorești ca un optimizator global să găsească un set bun de unghiuri, iar apoi un optimizator mai fin să urmeze un gradient către cel mai bun set de unghiuri vecine. În cazul dinamicii computaționale a fluidelor (CFD), secvența de optimizare implicită produce cele mai bune rezultate — dar în cazul deformării materialelor (MD), deși valorile implicite oferă rezultate bune, le poți configura mai detaliat pentru beneficii specifice problemei.

Reține că, pentru fiecare variabilă a funcției, specificăm numărul de qubiți (cu care poți experimenta). Prin stivuirea a 10 circuite identice și evaluarea celor 10 observabile identice pe qubiți diferiți de-a lungul unui circuit mare, poți aplica atenuarea zgomotului în cadrul procesului de optimizare CMA, bazându-te pe metoda de învățare a zgomotului, și reduce semnificativ numărul de shot-uri necesare.
## Intrare/ieșire
### Dinamica Computațională a Fluidelor
Ecuația Burgers fără vâscozitate modelează curgerea fluidelor nevâscoase după cum urmează:

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = 0,$$

$u$ reprezintă câmpul vitezei fluidului. Acest caz de utilizare are o condiție la limită temporală: poți selecta condiția inițială și apoi permite sistemului să se relaxeze. În prezent, singurele condiții inițiale acceptate sunt funcțiile liniare: $ax + b$.

Argumentele pentru ecuațiile diferențiale ale CFD se află pe o grilă fixă, după cum urmează:

- $t$ este între 0 și 0.95 cu 30 de puncte de eșantionare. $x$ este între 0 și 0.95 cu un pas de 0.2375.

### Deformarea Materialelor
Acest caz de utilizare se concentrează pe deformarea hipoelastică cu testul de tracțiune unidimensional, în care o bară fixată în spațiu este trasă la celălalt capăt. Descriem problema după cum urmează:

$$u' - \frac{\sigma}{3K} - \frac{2}{\sqrt{3}}\epsilon_0\left(\frac{\sigma'}{\sigma_0\sqrt{3}}\right)^n = 0$$

$$\sigma' - b = 0,$$

$K$ reprezintă modulul de compresibilitate al materialului întins, $n$ exponentul unei legi de putere, $b$ forța pe unitate de masă, $\epsilon_0$ limita de stres proporțional, $\sigma_0$ limita de deformare proporțională, $u$ funcția de stres și $\sigma$ funcția de deformare.

Bara considerată are lungime unitară. Acest caz de utilizare are o condiție la limită pentru stresul de suprafață $t$, adică cantitatea de lucru necesară pentru a întinde bara.

Argumentele pentru ecuațiile diferențiale ale MD se află pe o grilă fixă, după cum urmează:

- $x$ este între 0 și 1 cu un pas de 0.04.
### Intrare
Pentru a rula Funcția Qiskit QUICK-PDE, poți ajusta următorii parametri:

| Nume              | Tip                                                  | Descriere                                                                                                                                                                                                                                                                       | Specific cazului de utilizare | Exemplu                  |
| ----------------- | ---------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------- | ------------------------ |
| use_case          | `Literal["MD", "CFD"]`                               | Comutator pentru a selecta sistemul de ecuații diferențiale de rezolvat                                                                                                                                                                                                         | Nu                            | `"CFD"`                  |
| parameters        | `dict[str, Any]`                                     | Parametrii ecuației diferențiale (vezi tabelul următor pentru mai multe detalii)                                                                                                                                                                                                | Nu                            | `{"a": 1.0, "b": 1.0}`   |
| nb_qubits         | `Optional[dict[str, dict[str, int]]]`                | Numărul de qubiți per funcție și per variabilă. Valorile optimizate sunt alese de funcție, dar dacă vrei să găsești o combinație mai bună, poți suprascrie valorile implicite                                                                                                    | Nu                            | `{"u": {"x": 1, "t":3}}` |
| depth             | `Optional[dict[str, int]]`                           | Adâncimea ansatz-ului per funcție. Valorile optimizate sunt alese de funcție, dar dacă vrei să găsești o combinație mai bună, poți suprascrie valorile implicite                                                                                                                | Nu                            | `{"u": 4}`               |
| optimizer         | `Optional[list[str]]`                                | Optimizatorii de utilizat, fie "CMAES" din [biblioteca python `cma`](https://github.com/CMA-ES/pycma) fie unul dintre [optimizatorii scipy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)                                                   | MD                            | `"SLSQP"`                |
| shots             | `Optional[list[int]]`                                | Numărul de shot-uri utilizate pentru a rula fiecare circuit. Deoarece sunt necesari mai mulți pași de optimizare, lungimea listei trebuie să fie egală cu numărul de optimizatori utilizați (4 în cazul CFD). Implicit `[50_000] * nb_optimizers` pentru MD și `[5_00, 2_000, 5_000, 10_000]` pentru CFD | Nu                            | `[15_000, 30_000]`       |
| optimizer_options | `Optional[dict[str, Any]]`                           | Opțiuni de transmis optimizatorului. Detaliile acestei intrări depind de optimizatorul utilizat; pentru specificații, consultă documentația optimizatorului folosit                                                                                                              | MD                            | `{"maxiter": 50 }`       |
| initialization    | `Optional[Literal["RANDOM", "PHYSICALLY_INFORMED"]]` | Dacă să se înceapă cu unghiuri aleatoare sau cu unghiuri alese inteligent. Reține că unghiurile alese inteligent pot să nu funcționeze în 100% din cazuri. Implicit `"PHYSICALLY_INFORMED"`.                                                                                     | Nu                            | `"RANDOM"`               |
| backend_name      | `Optional[str]`                                      | Numele Backend-ului de utilizat.                                                                                                                                                                                                                                                | Nu                            | `"ibm_torino"`           |
| mode              | `Optional[Literal["job", "session", "batch"]]`        | Modul de execuție de utilizat. Implicit `"job"`.                                                                                                                                                                                                                                | Nu                            | `"job"`                  |

Parametrii ecuației diferențiale (parametri fizici și condiție la limită) trebuie să urmeze formatul dat:

| Caz de utilizare | Cheie       | Tip valoare | Descriere                                    | Exemplu |
| ---------------- | ----------- | ----------- | -------------------------------------------- | ------- |
| CFD              | `a`         | `float`     | Coeficientul valorilor inițiale ale lui $u$  | `1.0`   |
| CFD              | `b`         | `float`     | Decalajul valorilor inițiale ale lui $u$     | `1.0`   |
| MD               | `t`         | `float`     | stres de suprafață                           | `12.0`  |
| MD               | `K`         | `float`     | modul de compresibilitate                    | `100.0` |
| MD               | `n`         | `int`       | lege de putere                               | `4.0`   |
| MD               | `b`         | `float`     | forță pe unitate de masă                     | `10.0`  |
| MD               | `epsilon_0` | `float`     | limită de stres proporțional                 | `0.1`   |
| MD               | `sigma_0`   | `float`     | limită de deformare proporțională            | `5.0`   |
### Ieșire
Ieșirea este un dicționar cu lista de puncte de eșantionare, precum și valorile funcțiilor în fiecare dintre aceste puncte:

In [ ]:
from numpy import array

In [ ]:
solution = {
    "functions": {
        "u": array(
            [
                [0.01, 0.1, 1],
                [0.02, 0.2, 2],
                [0.03, 0.3, 3],
                [0.04, 0.4, 4],
            ]
        ),
    },
    "samples": {
        "t": array([0.1, 0.2, 0.3, 0.4]),
        "x": array([0.5, 0.6, 0.7]),
    },
}

Forma unui array de soluție depinde de eșantioanele variabilelor:

In [ ]:
assert len(solution["functions"]["u"].shape) == len(solution["samples"])
for col_size, samples in zip(
    solution["functions"]["u"].shape, solution["samples"].values()
):
    assert col_size == len(samples)

Maparea dintre punctele de eșantionare ale variabilelor funcției și dimensiunea array-ului de soluție se face în ordine alfanumerică a numelui variabilei. De exemplu, dacă variabilele sunt `"t"` și `"x"`, un rând din `solution["functions"]["u"]` reprezintă valorile soluției pentru un `"t"` fix, iar o coloană din `solution["functions"]["u"]` reprezintă valorile soluției pentru un `"x"` fix.

Urmează un exemplu despre cum să obții valoarea funcției pentru un set specific de coordonate:

In [ ]:
# u(t=0.2, x=0.7) == 2
assert solution["samples"]["t"][1] == 0.2
assert solution["samples"]["x"][2] == 0.7
assert solution["functions"]["u"][1, 2] == 2

## Benchmark-uri

Tabelul următor prezintă statistici pentru diverse rulări ale funcției noastre.

| Exemplu                        | Număr de qubiți | Inițializare          | Eroare    | Timp total (min) | Utilizare runtime (min) |
| ------------------------------ | ---------------- | --------------------- | --------- | ---------------- | ----------------------- |
| Ecuația Burgers fără vâscozitate | 50             | `PHYSICALLY_INFORMED` | $10^{-2}$ | 66               | 25                      |
| Test de tracțiune hipoelastic 1D | 18             | `RANDOM`              | $10^{-2}$ | 123              | 100                     |

## Primii pași

Completează [formularul pentru a solicita acces la funcția QUICK-PDE.](https://forms.cloud.microsoft/e/3Wi9cbjQPK) Apoi, presupunând că ți-ai [salvat deja contul](/guides/functions#install-qiskit-functions-catalog-client) în mediul local, selectează funcția după cum urmează:

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

quick = catalog.load("colibritd/quick-pde")

## Exemple

Pentru a începe, încearcă unul dintre următoarele exemple:

### Dinamica Computațională a Fluidelor (CFD)

Când condițiile inițiale sunt setate la $u(0,x) = x$, rezultatele sunt după cum urmează:

In [ ]:
# launch the simulation with initial conditions u(0,x) = a*x + b
job = quick.run(use_case="cfd", physical_parameters={"a": 1.0, "b": 0.0})

Verifică [statusul](/guides/functions#check-job-status) sarcinii de lucru a Funcției Qiskit sau returnează [rezultatele](/guides/functions#retrieve-results) după cum urmează:

In [ ]:
print(job.status())
solution = job.result()

'QUEUED'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

_ = plt.figure()
ax = plt.axes(projection="3d")

# plot the solution using the 3d plotting capabilities of pyplot
t, x = np.meshgrid(solution["samples"]["t"], solution["samples"]["x"])
ax.plot_surface(
    t,
    x,
    solution["functions"]["u"],
    edgecolor="royalblue",
    lw=0.25,
    rstride=26,
    cstride=26,
    alpha=0.3,
)
ax.scatter(t, x, solution["functions"]["u"], marker=".")
ax.set(xlabel="t", ylabel="x", zlabel="u(t,x)")

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif" alt="Output of the previous code cell" />

### Material Deformation

The material deformation use case requires the physical parameters of your material and the applied force, as follows:

In [ ]:
import matplotlib.pyplot as plt

# select the properties of your material
job = quick.run(
    use_case="md",
    physical_parameters={
        "t": 12.0,
        "K": 100.0,
        "n": 4.0,
        "b": 10.0,
        "epsilon_0": 0.1,
        "sigma_0": 5.0,
    },
)

# plot the result
solution = job.result()

_ = plt.figure()
stress_plot = plt.subplot(211)
plt.plot(solution["samples"]["x"], solution["functions"]["u"])
strain_plot = plt.subplot(212)
plt.plot(solution["samples"]["x"], solution["functions"]["sigma"])

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif" alt="Output of the previous code cell" />

![Ieșirea celulei de cod anterioare](../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif)

### Deformarea Materialelor
Cazul de utilizare al deformării materialelor necesită parametrii fizici ai materialului tău și forța aplicată, după cum urmează:

In [ ]:
job = quick.run(use_case="mdf", physical_params={})

print(job.error_message())


# or write a wrapper around it for a more human readable version
def pprint_error(job):
    print("".join(eval(job.error_message())["error"]))


print("___")
pprint_error(job)

{"error": ["qiskit.exceptions.QiskitError: 'Unknown argument \"physical_params\", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'\n"]}
___
qiskit.exceptions.QiskitError: 'Unknown argument "physical_params", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'



![Ieșirea celulei de cod anterioare](../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif)

## Obține mesaje de eroare

Dacă statusul sarcinii tale de lucru este `ERROR`, folosește `job.error_message()` pentru a obține mesajul de eroare care te ajută la depanare, după cum urmează: